# PCC170 – Entrega 2: Fatorial 2³ r com Bootstrap (STEM + Junior)
Fatores: **A** uso de IA (Passivo +1 / Ativo −1), **B** ansiedade (≥Q3 = +1 / <Q1 = −1),
**C** horas de estudo (≥Q3 = +1 / <Q1 = −1). Resposta: ΔGPA = Post − Pre.

In [1]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import statsmodels.api as sm, statsmodels.formula.api as smf
from scipy import stats
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

CSV = "ai_student_impact_dataset.csv"
if not os.path.exists(CSV):
    for p in ("../csv/ai_student_impact_dataset.csv", "csv/ai_student_impact_dataset.csv"):
        if os.path.exists(p):
            CSV = p
            break
OUT_FIG, OUT_DAT = "figuras", "dados"
os.makedirs(OUT_FIG, exist_ok=True); os.makedirs(OUT_DAT, exist_ok=True)

SEED, R, M, NBOOT = 42, 30, 36, 5000
PASSIVO = ["Copywriting/Drafting", "Direct_Answer_Generation"]
CELLS = [(a, b, c) for c in (-1, 1) for b in (-1, 1) for a in (-1, 1)]  # ordem de Yates
EFFECTS = {"A": lambda k: k[0], "B": lambda k: k[1], "C": lambda k: k[2],
           "AB": lambda k: k[0]*k[1], "AC": lambda k: k[0]*k[2],
           "BC": lambda k: k[1]*k[2], "ABC": lambda k: k[0]*k[1]*k[2]}

def save(name):
    plt.tight_layout(); plt.savefig(f"{OUT_FIG}/{name}.png", dpi=200); plt.show()

## 1. Dados: ΔGPA, recorte STEM + Junior, quartis e níveis ±1

In [2]:
df = pd.read_csv(CSV)
df["Delta_GPA"] = df["Post_Semester_GPA"] - df["Pre_Semester_GPA"]
df["A"] = np.where(df["Primary_Use_Case"].isin(PASSIVO), 1, -1)
s = df[(df.Major_Category == "STEM") & (df.Year_of_Study == "Junior")].copy()
print(f"Dataset: {len(df)} | STEM+Junior: {len(s)}")

an, hr = s["Anxiety_Level_During_Exams"], s["Traditional_Study_Hours"]
qa1, qa3 = an.quantile(.25), an.quantile(.75)
qh1, qh3 = hr.quantile(.25), hr.quantile(.75)
print(f"Ansiedade Q1={qa1} Q3={qa3} | Horas Q1={qh1:.2f} Q3={qh3:.3f}")

s["B"] = np.where(an >= qa3, 1, np.where(an < qa1, -1, 0))
s["C"] = np.where(hr >= qh3, 1, np.where(hr < qh1, -1, 0))
fat = s[(s.B != 0) & (s.C != 0)].copy()
fat["celula"] = [CELLS.index(k) + 1 for k in zip(fat.A, fat.B, fat.C)]
fat.to_csv(f"{OUT_DAT}/df_factorial.csv", index=False)
print(f"Alunos nos níveis extremos: {len(fat)} ({len(fat)/len(s):.1%} do recorte)")

tab = fat.groupby(["celula", "A", "B", "C"]).Delta_GPA.agg(["count", "mean", "std"]).round(3)
print(tab)
G = {k: v["Delta_GPA"].values for k, v in fat.groupby(["A", "B", "C"])}
assert len(G) == 8 and min(len(v) for v in G.values()) >= max(M, 10)

Dataset: 50000 | STEM+Junior: 3279
Ansiedade Q1=3.0 Q3=6.0 | Horas Q1=7.33 Q3=14.475
Alunos nos níveis extremos: 848 (25.9% do recorte)
                 count   mean    std
celula A  B  C                      
1      -1 -1 -1    133  0.164  0.180
2       1 -1 -1     36  0.114  0.165
3      -1  1 -1    205  0.157  0.180
4       1  1 -1     63  0.028  0.250
5      -1 -1  1    118  0.326  0.155
6       1 -1  1     49  0.334  0.155
7      -1  1  1    187  0.340  0.178
8       1  1  1     57  0.243  0.189


### Balanceamento de Pre_GPA e descritivas

In [3]:
bal = fat.groupby("celula").Pre_Semester_GPA.agg(["mean", "std"]).round(3)
print(bal)
print("ANOVA Pre_GPA ~ célula: p =", round(stats.f_oneway(*[g.Pre_Semester_GPA for _, g in fat.groupby("celula")]).pvalue, 4))
desc = df[["Delta_GPA", "Anxiety_Level_During_Exams", "Traditional_Study_Hours"]].agg(["mean", "std", "min", "max"]).T.round(2)
desc_s = s[["Delta_GPA", "Anxiety_Level_During_Exams", "Traditional_Study_Hours"]].agg(["mean", "std", "min", "max"]).T.round(2)
desc.to_csv(f"{OUT_DAT}/descritivas_dataset.csv"); desc_s.to_csv(f"{OUT_DAT}/descritivas_stem_junior.csv")
print(desc, "\n", desc_s)

         mean    std
celula              
1       3.056  0.542
2       3.196  0.408
3       3.161  0.460
4       3.144  0.521
5       3.139  0.468
6       3.143  0.456
7       3.148  0.489
8       3.180  0.429
ANOVA Pre_GPA ~ célula: p = 0.6136
                             mean   std   min    max
Delta_GPA                    0.20  0.19 -0.92   1.01
Anxiety_Level_During_Exams   4.27  2.14  1.00  10.00
Traditional_Study_Hours     11.21  5.16  1.00  35.86 
                              mean   std   min    max
Delta_GPA                    0.24  0.19 -0.85   0.89
Anxiety_Level_During_Exams   4.42  2.23  1.00  10.00
Traditional_Study_Hours     11.04  5.29  1.00  29.75


## 2. Gráficos exploratórios

In [4]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
sns.histplot(s.Delta_GPA, kde=True, ax=ax[0], color="#2980b9"); ax[0].set_title("ΔGPA (STEM Junior)")
sns.histplot(an, discrete=True, ax=ax[1], color="#e67e22")
for q, c in ((qa1, "green"), (qa3, "red")): ax[1].axvline(q, color=c, ls="--", label=f"Q={q:g}")
ax[1].set_title("Ansiedade com Q1/Q3"); ax[1].legend()
sns.histplot(hr, kde=True, ax=ax[2], color="#8e44ad")
for q, c in ((qh1, "green"), (qh3, "red")): ax[2].axvline(q, color=c, ls="--", label=f"Q={q:.2f}")
ax[2].set_title("Horas de estudo com Q1/Q3"); ax[2].legend()
save("01_distribuicoes_quartis")

In [5]:
n_tab = fat.groupby(["A", "B", "C"]).size().unstack("C")
plt.figure(figsize=(6, 4)); sns.heatmap(n_tab, annot=True, fmt="d", cmap="Blues")
plt.title("n por célula (linhas A,B / colunas C)"); save("02_n_por_celula")

In [6]:
fat["rotulo"] = fat.celula.astype(str) + " (" + fat.A.map({-1: "At", 1: "Ps"}) + "," + fat.B.map({-1: "B", 1: "A"}) + "," + fat.C.map({-1: "b", 1: "a"}) + ")"
plt.figure(figsize=(12, 5)); sns.boxplot(data=fat.sort_values("celula"), x="rotulo", y="Delta_GPA", palette="viridis")
plt.xticks(rotation=30); plt.xlabel("Célula (IA, Ansiedade, Estudo)"); save("03_boxplot_bruto_por_celula")

### CDF empírica dos dados analisados

In [7]:
def ecdf(x):
    x = np.sort(np.asarray(x)); return x, np.arange(1, len(x) + 1) / len(x)

rot_a = {-1: "Ativo", 1: "Passivo"}
fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
for lvl, g in fat.groupby("A"):
    ax[0].step(*ecdf(g.Delta_GPA), where="post", label=f"{rot_a[lvl]} (n={len(g)})")
ax[0].set_xlabel("ΔGPA"); ax[0].set_title("CDF do ΔGPA por tipo de uso de IA (n=848)")
for lvl, g in fat.groupby("B"):
    ax[1].step(*ecdf(g.Delta_GPA), where="post", label=f"Ansiedade {'alta' if lvl == 1 else 'baixa'} (n={len(g)})")
ax[1].set_xlabel("ΔGPA"); ax[1].set_title("CDF do ΔGPA por nível de ansiedade")
for lvl, g in fat.groupby("C"):
    ax[2].step(*ecdf(g.Delta_GPA), where="post", label=f"Estudo {'alto' if lvl == 1 else 'baixo'} (n={len(g)})")
ax[2].set_xlabel("ΔGPA"); ax[2].set_title("CDF do ΔGPA por nível de horas de estudo")
for a_ in ax: a_.set_ylabel("F(x) = P(ΔGPA ≤ x)"); a_.legend(loc="lower right")
save("15_cdf_fatores")

In [8]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
for i, k in enumerate(CELLS):
    lab = f"{i+1}: {rot_a[k[0]][:2]}, Ans{'+' if k[1] == 1 else '-'}, Est{'+' if k[2] == 1 else '-'} (n={len(G[k])})"
    ax[0].step(*ecdf(G[k]), where="post", label=lab)
ax[0].set_title("CDF do ΔGPA bruto por célula 2³"); ax[0].set_xlabel("ΔGPA")
ax[1].set_title("CDF – recorte STEM Junior completo vs. 848 analisados")
ax[1].step(*ecdf(s.Delta_GPA), where="post", label=f"STEM Junior (n={len(s)})")
ax[1].step(*ecdf(fat.Delta_GPA), where="post", label=f"Extremos Q1/Q3 (n={len(fat)})")
ax[1].set_xlabel("ΔGPA")
for a_ in ax: a_.set_ylabel("F(x)"); a_.legend(fontsize=8, loc="lower right")
save("16_cdf_por_celula_e_recorte")

In [9]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
for var, nome, q1, q3, a_ in ((an, "Ansiedade", qa1, qa3, ax[0]), (hr, "Horas de estudo", qh1, qh3, ax[1])):
    a_.step(*ecdf(var), where="post", color="#2c3e50")
    a_.axvline(q1, color="green", ls="--", label=f"Q1={q1:.2f} (F=0,25)"); a_.axvline(q3, color="red", ls="--", label=f"Q3={q3:.2f} (F=0,75)")
    a_.axhline(.25, color="gray", lw=.6); a_.axhline(.75, color="gray", lw=.6)
    a_.set_xlabel(nome); a_.set_ylabel("F(x)"); a_.set_title(f"CDF – {nome} (STEM Junior) com cortes"); a_.legend()
save("17_cdf_ansiedade_horas_quartis")

## 3. Ciclo 1 – Réplicas bootstrap (r × 8 células, m alunos com reposição) + ANOVA 2³

In [10]:
def make_replicas(G, r, m, seed):
    rng = np.random.default_rng(seed); rows = []
    for k in CELLS:
        for j in range(r):
            idx = rng.integers(0, len(G[k]), m)
            rows.append(dict(A=k[0], B=k[1], C=k[2], celula=CELLS.index(k) + 1, rep=j + 1,
                             seed=seed, Y=G[k][idx].mean(), sd_amostra=G[k][idx].std(ddof=1),
                             distintos=len(set(idx))))
    return pd.DataFrame(rows)

def effects_from_means(mu):
    return {n: sum(f(k) * mu[k] for k in CELLS) / 4 for n, f in EFFECTS.items()}

rep = make_replicas(G, R, M, SEED)
rep.to_csv(f"{OUT_DAT}/replicas.csv", index=False)
print(rep.shape, "| mín. de alunos distintos por réplica:", rep.distintos.min())

mod = smf.ols("Y ~ A*B*C", rep).fit()
anova = sm.stats.anova_lm(mod, typ=2)
anova["pct_var"] = anova.sum_sq / anova.sum_sq.sum() * 100
anova.to_csv(f"{OUT_DAT}/anova_bootstrap.csv"); print(anova.round(5))
ef_anova = {n.replace(":", ""): 2 * mod.params[n] for n in mod.params.index if n != "Intercept"}

(240, 9) | mín. de alunos distintos por réplica: 20
           sum_sq     df           F   PR(>F)   pct_var
A         0.27269    1.0   279.76225  0.00000   8.94438
B         0.10010    1.0   102.69709  0.00000   3.28337
A:B       0.13213    1.0   135.55492  0.00000   4.33388
C         2.27792    1.0  2337.00063  0.00000  74.71710
A:C       0.03126    1.0    32.06573  0.00000   1.02519
B:C       0.00308    1.0     3.16160  0.07670   0.10108
A:B:C     0.00542    1.0     5.55654  0.01924   0.17765
Residual  0.22613  232.0         NaN      NaN   7.41736


### Validação das premissas (Ciclo 1)

In [11]:
res = mod.resid
sh = stats.shapiro(res); lev = stats.levene(*[g.Y.values for _, g in rep.groupby("celula")])
# normalidade dos dados brutos x médias bootstrap, célula a célula
norm_tab = pd.DataFrame({
    "n": [len(G[k]) for k in CELLS],
    "skew_bruto": [stats.skew(G[k]) for k in CELLS],
    "shapiro_p_bruto": [stats.shapiro(G[k]).pvalue for k in CELLS],
    "skew_boot": [stats.skew(rep[rep.celula == i + 1].Y) for i in range(8)],
    "shapiro_p_boot": [stats.shapiro(rep[rep.celula == i + 1].Y).pvalue for i in range(8)]}, index=range(1, 9)).round(3)
norm_tab.to_csv(f"{OUT_DAT}/normalidade_por_celula.csv"); print(norm_tab)
print(f"Shapiro resíduos: p={sh.pvalue:.3f} ({'OK' if sh.pvalue > .05 else 'FALHOU'})")
print(f"Levene: p={lev.pvalue:.3f} ({'OK' if lev.pvalue > .05 else 'FALHOU'})")

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
sm.qqplot(res, line="45", fit=True, ax=ax[0]); ax[0].set_title("QQ dos resíduos (Y bootstrap)")
ax[1].scatter(mod.fittedvalues, res, alpha=.5); ax[1].axhline(0, color="r"); ax[1].set_title("Resíduos × ajustados")
ax[1].set_xlabel("Ajustado"); ax[1].set_ylabel("Resíduo")
sns.histplot(res, kde=True, ax=ax[2]); ax[2].set_title("Histograma dos resíduos")
save("04_diagnostico_residuos")

     n  skew_bruto  shapiro_p_bruto  skew_boot  shapiro_p_boot
1  133       0.164            0.223      0.092           0.929
2   36      -0.221            0.799     -0.274           0.804
3  205       0.003            0.297     -0.395           0.461
4   63      -0.677            0.067     -0.050           0.902
5  118       0.330            0.103      0.018           0.614
6   49       0.119            0.949      0.535           0.101
7  187       0.126            0.139     -0.079           0.755
8   57      -0.686            0.052      0.299           0.683
Shapiro resíduos: p=0.807 (OK)
Levene: p=0.119 (OK)


In [12]:
cores = sns.color_palette("tab10", 8)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
sm.qqplot(np.concatenate([(G[k] - G[k].mean()) / G[k].std() for k in CELLS]), line="45", ax=ax[0]); ax[0].set_title("QQ – ΔGPA bruto padronizado por célula")
sm.qqplot(np.concatenate([(g.Y - g.Y.mean()) / g.Y.std() for _, g in rep.groupby("celula")]), line="45", ax=ax[1]); ax[1].set_title("QQ – médias bootstrap padronizadas")
save("05_qq_antes_depois")

fig, axs = plt.subplots(2, 4, figsize=(16, 6), sharex=True)
for i, ax_ in enumerate(axs.ravel()):
    sns.histplot(rep[rep.celula == i + 1].Y, kde=True, ax=ax_, color=cores[i]); ax_.set_title(f"Célula {i+1} {CELLS[i]}", fontsize=10)
save("06_hist_bootstrap_por_celula")

In [13]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
for i, k in enumerate(CELLS):
    ax[0].step(*ecdf(rep[rep.celula == i + 1].Y), where="post", label=f"Célula {i+1}")
ax[0].set_title("CDF das médias bootstrap (Y) por célula"); ax[0].set_xlabel("Y = média de ΔGPA"); ax[0].legend(fontsize=8)
for i in range(8):
    x, f = ecdf(rep[rep.celula == i + 1].Y); mu_, sd_ = x.mean(), x.std(ddof=1)
    ax[1].plot(np.sort((x - mu_) / sd_), f, lw=1, alpha=.7)
z = np.linspace(-3, 3, 200); ax[1].plot(z, stats.norm.cdf(z), "k--", lw=2, label="Normal(0,1)")
ax[1].set_title("CDF das médias bootstrap padronizadas × Normal"); ax[1].set_xlabel("z"); ax[1].legend()
for a_ in ax: a_.set_ylabel("F(x)")
save("18_cdf_bootstrap")

## Quadros do desenho: matriz de sinais e combinações de teste

In [14]:
nomes_a = {-1: "Ativo", 1: "Passivo"}; nomes_b = {-1: "Baixa (<Q1)", 1: "Alta (≥Q3)"}; nomes_c = {-1: "Baixas (<Q1)", 1: "Altas (≥Q3)"}
yates = ["(1)", "a", "b", "ab", "c", "ac", "bc", "abc"]
sinais = pd.DataFrame([{"Ensaio": i + 1, "Yates": yates[i], "I": 1, **{n: f(k) for n, f in EFFECTS.items()}} for i, k in enumerate(CELLS)])
sinais["Y_média_obs"] = [G[k].mean() for k in CELLS]
sinais["Y_média_boot"] = [rep[rep.celula == i + 1].Y.mean() for i in range(8)]
linha = {"Ensaio": "Divisor", "Yates": "", "I": 8, **{n: 4 for n in EFFECTS}}
sinais_out = pd.concat([sinais, pd.DataFrame([linha])], ignore_index=True)
ef_obs_q = effects_from_means({k: G[k].mean() for k in CELLS})
ef_lin = {"Ensaio": "Efeito", "Yates": "", **ef_obs_q}
sinais_out = pd.concat([sinais_out, pd.DataFrame([ef_lin])], ignore_index=True)
sinais_out.to_csv(f"{OUT_DAT}/quadro_sinais.csv", index=False); print(sinais_out.round(4).to_string(index=False))

comb = pd.DataFrame([{"Ensaio": i + 1, "Yates": yates[i],
    "A (IA)": f"{k[0]:+d} {nomes_a[k[0]]}", "B (Ansiedade)": f"{k[1]:+d} {nomes_b[k[1]]}", "C (Estudo)": f"{k[2]:+d} {nomes_c[k[2]]}",
    "n alunos": len(G[k]), "ΔGPA médio": round(G[k].mean(), 3), "DP": round(G[k].std(ddof=1), 3),
    "Réplicas (r)": R, "m por réplica": M, "Observações": R} for i, k in enumerate(CELLS)])
comb.to_csv(f"{OUT_DAT}/quadro_combinacoes.csv", index=False); print(comb.to_string(index=False))

fig, ax = plt.subplots(2, 1, figsize=(13, 8.5)); 
for a_ in ax: a_.axis("off")
t1 = ax[0].table(cellText=comb.astype(str).values, colLabels=comb.columns, loc="center", cellLoc="center"); t1.auto_set_font_size(False); t1.set_fontsize(8); t1.scale(1, 1.5)
ax[0].set_title("Quadro das combinações de teste (8 tratamentos × r = 30 réplicas)", fontweight="bold")
sg = sinais[["Ensaio", "Yates", "I", *EFFECTS, "Y_média_obs"]].copy(); sg["Y_média_obs"] = sg["Y_média_obs"].round(3)
t2 = ax[1].table(cellText=sg.astype(str).values, colLabels=sg.columns, loc="center", cellLoc="center"); t2.auto_set_font_size(False); t2.set_fontsize(9); t2.scale(1, 1.5)
for (r_, c_), cell in t2.get_celld().items():
    if r_ > 0 and 2 <= c_ <= 9:
        cell.set_facecolor("#d5f5e3" if cell.get_text().get_text() == "1" else "#fadbd8")
ax[1].set_title("Matriz de sinais dos contrastes (efeito = Σ sinal·Y / 4)", fontweight="bold")
save("19_quadros_combinacoes_e_sinais")

 Ensaio Yates   I       A       B       C     AB      AC     BC     ABC  Y_média_obs  Y_média_boot
      1   (1) 1.0 -1.0000 -1.0000 -1.0000  1.000  1.0000  1.000 -1.0000       0.1638        0.1668
      2     a 1.0  1.0000 -1.0000 -1.0000 -1.000 -1.0000  1.000  1.0000       0.1139        0.1140
      3     b 1.0 -1.0000  1.0000 -1.0000 -1.000  1.0000 -1.000  1.0000       0.1567        0.1563
      4    ab 1.0  1.0000  1.0000 -1.0000  1.000 -1.0000 -1.000 -1.0000       0.0278        0.0286
      5     c 1.0 -1.0000 -1.0000  1.0000  1.000 -1.0000 -1.000  1.0000       0.3263        0.3222
      6    ac 1.0  1.0000 -1.0000  1.0000 -1.000  1.0000 -1.000 -1.0000       0.3338        0.3340
      7    bc 1.0 -1.0000  1.0000  1.0000 -1.000 -1.0000  1.000 -1.0000       0.3402        0.3450
      8   abc 1.0  1.0000  1.0000  1.0000  1.000  1.0000  1.000  1.0000       0.2427        0.2439
Divisor       8.0  4.0000  4.0000  4.0000  4.000  4.0000  4.000  4.0000          NaN           NaN
 Efeito   

## 4. Efeitos, cubo, interações e Pareto

In [15]:
mu = {k: rep[rep.celula == i + 1].Y.mean() for i, k in enumerate(CELLS)}
ef_yates = effects_from_means(mu)
print({k: round(v, 4) for k, v in ef_yates.items()})

fig = plt.figure(figsize=(7, 6)); ax = fig.add_subplot(111, projection="3d")
for k in CELLS:
    ax.scatter(*k, s=900 * abs(mu[k]) + 100, c=[mu[k]], cmap="RdYlGn", vmin=0, vmax=.45)
    ax.text(k[0], k[1], k[2] + .08, f"{mu[k]:.3f}", ha="center")
ax.set_xlabel("A (IA)"); ax.set_ylabel("B (Ansiedade)"); ax.set_zlabel("C (Estudo)")
ax.set_xticks([-1, 1]); ax.set_yticks([-1, 1]); ax.set_zticks([-1, 1]); ax.set_title("Cubo 2³ – média de ΔGPA")
save("07_cubo_2x3")

{'A': np.float64(-0.0674), 'B': np.float64(-0.0408), 'C': np.float64(0.1948), 'AB': np.float64(-0.0469), 'AC': np.float64(0.0228), 'BC': np.float64(0.0072), 'ABC': np.float64(-0.0095)}


In [16]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for i, (f, nome) in enumerate((("A", "IA (Ativo → Passivo)"), ("B", "Ansiedade"), ("C", "Estudo"))):
    m_ = rep.groupby(f).Y.agg(["mean", "std"])
    ax[i].errorbar([-1, 1], m_["mean"], yerr=1.96 * m_["std"] / np.sqrt(rep.groupby(f).size()), marker="o", capsize=4)
    ax[i].set_xticks([-1, 1]); ax[i].set_title(f"Efeito principal: {nome}"); ax[i].set_ylabel("ΔGPA médio")
save("08_efeitos_principais")

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for i, (f1, f2) in enumerate((("A", "B"), ("A", "C"), ("B", "C"))):
    for lvl, g in rep.groupby(f2):
        m_ = g.groupby(f1).Y.mean(); ax[i].plot(m_.index, m_.values, marker="o", label=f"{f2}={lvl}")
    ax[i].set_xticks([-1, 1]); ax[i].set_xlabel(f1); ax[i].set_ylabel("ΔGPA médio"); ax[i].set_title(f"Interação {f1}×{f2}"); ax[i].legend()
save("09_interacoes")

In [17]:
ef_s = pd.Series(ef_yates).reindex(["A", "B", "C", "AB", "AC", "BC", "ABC"])
sd_res = np.sqrt(mod.scale)
pa = (ef_s / sd_res).abs().sort_values()
plt.figure(figsize=(7, 4)); plt.barh(pa.index, pa.values, color=["#c0392b" if ef_s[i] < 0 else "#27ae60" for i in pa.index])
plt.title("Pareto dos efeitos padronizados (|efeito|/DP resíduo)\nverde: positivo | vermelho: negativo"); save("10_pareto_efeitos")

order = ef_s.sort_values().index; q = stats.norm.ppf((np.arange(1, 8) - .5) / 7)
plt.figure(figsize=(6, 4.5)); plt.scatter(ef_s[order], q)
for n_, x_, y_ in zip(order, ef_s[order], q): plt.annotate(n_, (x_, y_), xytext=(4, 3), textcoords="offset points")
plt.xlabel("Efeito"); plt.ylabel("Quantil normal"); plt.title("Normal plot dos efeitos"); save("11_normal_plot_efeitos")

## 5. Ciclo 2 – Bootstrap estratificado dos efeitos (decisão de significância)

In [18]:
def boot_effects(G, nboot, seed):
    rng = np.random.default_rng(seed); out = []
    for _ in range(nboot):
        out.append(effects_from_means({k: v[rng.integers(0, len(v), len(v))].mean() for k, v in G.items()}))
    return pd.DataFrame(out)

obs = effects_from_means({k: G[k].mean() for k in CELLS})
BS = boot_effects(G, NBOOT, SEED)
ci = pd.DataFrame({"efeito_obs": obs, "ic_inf": BS.quantile(.025), "ic_sup": BS.quantile(.975), "dp_boot": BS.std()})
ci["ic_exclui_0"] = (ci.ic_inf > 0) | (ci.ic_sup < 0)
print(ci.round(4))

fig, ax = plt.subplots(figsize=(7, 4.5))
for i, n_ in enumerate(ci.index[::-1]):
    c = "#c0392b" if ci.loc[n_, "ic_exclui_0"] else "gray"
    ax.errorbar(ci.loc[n_, "efeito_obs"], i, xerr=[[ci.loc[n_, "efeito_obs"] - ci.loc[n_, "ic_inf"]], [ci.loc[n_, "ic_sup"] - ci.loc[n_, "efeito_obs"]]], fmt="o", color=c, capsize=4)
ax.axvline(0, color="k", ls="--"); ax.set_yticks(range(7)); ax.set_yticklabels(ci.index[::-1])
ax.set_title("Forest plot: efeitos com IC95% bootstrap (vermelho = exclui 0)"); ax.set_xlabel("Efeito em ΔGPA"); save("12_forest_plot_efeitos")

     efeito_obs  ic_inf  ic_sup  dp_boot  ic_exclui_0
A       -0.0672 -0.0971 -0.0380   0.0149         True
B       -0.0426 -0.0710 -0.0133   0.0147         True
C        0.1952  0.1673  0.2246   0.0147         True
AB      -0.0460 -0.0759 -0.0172   0.0148         True
AC       0.0222 -0.0067  0.0523   0.0151        False
BC       0.0040 -0.0253  0.0343   0.0152        False
ABC     -0.0065 -0.0345  0.0235   0.0149        False


In [19]:
pv = anova.drop("Residual")
plt.figure(figsize=(7, 4)); sns.barplot(x=pv.index, y=pv.pct_var, color="#34495e"); plt.xticks(rotation=30)
plt.ylabel("% da variância (SS/SST)"); plt.title("Contribuição de cada termo (ANOVA bootstrap)"); save("13_pct_variancia")

## 6. Validação cruzada, sensibilidade e estabilidade

In [20]:
raw = sm.stats.anova_lm(smf.ols("Delta_GPA ~ A*B*C", fat).fit(), typ=2)
p_raw = raw["PR(>F)"].drop("Residual"); p_raw.index = [i.replace(":", "") for i in p_raw.index]

# estabilidade entre sementes
seeds = []
for sd in (42, 1, 2, 3, 4, 5, 6, 7, 8, 9):
    m_ = smf.ols("Y ~ A*B*C", make_replicas(G, R, M, sd)).fit()
    seeds.append({n.replace(":", ""): 2 * m_.params[n] for n in m_.params.index if n != "Intercept"})
seeds = pd.DataFrame(seeds); seeds.to_csv(f"{OUT_DAT}/seeds.csv", index=False)
est = seeds.agg(["mean", "std", "min", "max"]).T.round(4); print(est)

# sensibilidade nas regras de corte
def sens(regra):
    d = s.copy()
    if regra == "mediana":
        d["B"] = np.where(an > an.median(), 1, -1); d["C"] = np.where(hr > hr.median(), 1, -1)
    elif regra == "<=Q1/>=Q3":
        d["B"] = np.where(an >= qa3, 1, np.where(an <= qa1, -1, 0)); d["C"] = np.where(hr >= qh3, 1, np.where(hr <= qh1, -1, 0))
    else:
        d["B"] = s.B; d["C"] = s.C
    d = d[(d.B != 0) & (d.C != 0)]
    g = {k: v.Delta_GPA.values for k, v in d.groupby(["A", "B", "C"])}
    e = effects_from_means({k: g[k].mean() for k in CELLS})
    bs = boot_effects(g, 1000, SEED)
    return {"n": len(d), **{f"{k}": f"{e[k]:+.3f}{'*' if (bs[k].quantile(.025) > 0 or bs[k].quantile(.975) < 0) else ''}" for k in EFFECTS}}
sensib = pd.DataFrame({r: sens(r) for r in ("<Q1/>=Q3 (principal)", "<=Q1/>=Q3", "mediana")}).T
sensib.to_csv(f"{OUT_DAT}/sensibilidade.csv"); print(sensib, "\n(* = IC95% bootstrap exclui 0)")

# sensibilidade: ANCOVA com Pre_GPA
anc = smf.ols("Delta_GPA ~ A*B*C + Pre_Semester_GPA", fat).fit()
print("ANCOVA (Pre_GPA) coef A,B,C,AB:", (2 * anc.params[["A", "B", "C", "A:B"]]).round(4).to_dict())

       mean     std     min     max
A   -0.0674  0.0032 -0.0726 -0.0641
B   -0.0416  0.0055 -0.0479 -0.0325
AB  -0.0457  0.0013 -0.0481 -0.0439
C    0.1951  0.0049  0.1880  0.2023
AC   0.0239  0.0028  0.0171  0.0280
BC   0.0067  0.0021  0.0037  0.0095
ABC -0.0073  0.0034 -0.0124 -0.0026


                         n        A        B        C       AB      AC  \
<Q1/>=Q3 (principal)   848  -0.067*  -0.043*  +0.195*  -0.046*  +0.022   
<=Q1/>=Q3             1101  -0.080*  -0.038*  +0.193*  -0.033*  +0.011   
mediana               3279  -0.077*  -0.017*  +0.121*  -0.023*  -0.000   

                          BC     ABC  
<Q1/>=Q3 (principal)  +0.004  -0.006  
<=Q1/>=Q3             +0.006  +0.005  
mediana               +0.011  +0.010   
(* = IC95% bootstrap exclui 0)
ANCOVA (Pre_GPA) coef A,B,C,AB: {'A': -0.065, 'B': -0.0413, 'C': 0.196, 'A:B': -0.0478}


In [21]:
sub = sensib.drop(columns="n").map(lambda x: float(str(x).rstrip("*")))
sub.T.plot(kind="bar", figsize=(9, 4)); plt.axhline(0, color="k", lw=.8); plt.title("Sensibilidade: efeitos por regra de corte")
plt.ylabel("Efeito em ΔGPA"); plt.xticks(rotation=0); save("14_sensibilidade")

## 7. Consolidação e veredito

In [22]:
mestre = pd.DataFrame({
    "efeito": ci.efeito_obs, "ic95_inf": ci.ic_inf, "ic95_sup": ci.ic_sup,
    "efeito_anova_boot": pd.Series(ef_anova), "pct_var_boot": pv.pct_var.rename(lambda i: i.replace(":", "")),
    "p_anova_boot": pv["PR(>F)"].rename(lambda i: i.replace(":", "")), "p_anova_bruta": p_raw,
    "efeito_dp_entre_seeds": seeds.std()})
def veredito(r):
    a, b = r.ic95_inf > 0 or r.ic95_sup < 0, r.p_anova_bruta < .05
    return "significativo" if a and b else ("fraco" if a or b else "não significativo")
mestre["veredito"] = mestre.apply(veredito, axis=1)
mestre = mestre.reindex(["A", "B", "C", "AB", "AC", "BC", "ABC"])
mestre.round(4).to_csv(f"{OUT_DAT}/efeitos_ic.csv"); print(mestre.round(4).to_string())

     efeito  ic95_inf  ic95_sup  efeito_anova_boot  pct_var_boot  p_anova_boot  p_anova_bruta  efeito_dp_entre_seeds           veredito
A   -0.0672   -0.0971   -0.0380            -0.0674        8.9444        0.0000         0.0000                 0.0032      significativo
B   -0.0426   -0.0710   -0.0133            -0.0408        3.2834        0.0000         0.1269                 0.0055              fraco
C    0.1952    0.1673    0.2246             0.1948       74.7171        0.0000         0.0000                 0.0049      significativo
AB  -0.0460   -0.0759   -0.0172            -0.0469        4.3339        0.0000         0.0019                 0.0013      significativo
AC   0.0222   -0.0067    0.0523             0.0228        1.0252        0.0000         0.1522                 0.0028  não significativo
BC   0.0040   -0.0253    0.0343             0.0072        0.1011        0.0767         0.5649                 0.0021  não significativo
ABC -0.0065   -0.0345    0.0235            -0.00

In [23]:
checks = pd.DataFrame([
    ("Shapiro dos resíduos (p>0,05)", sh.pvalue > .05, f"p={sh.pvalue:.3f}"),
    ("Levene entre células (p>0,05)", lev.pvalue > .05, f"p={lev.pvalue:.3f}"),
    ("Estabilidade entre seeds (DP<0,01)", bool((seeds.std() < .01).all()), f"máx DP={seeds.std().max():.4f}"),
    ("Concordância IC × ANOVA bruta (sem 'fraco')", bool((mestre.veredito != "fraco").all()), ", ".join(mestre.index[mestre.veredito == "fraco"]) or "-"),
    ("Pre_GPA balanceado (ANOVA p>0,05)", stats.f_oneway(*[g.Pre_Semester_GPA for _, g in fat.groupby("celula")]).pvalue > .05, "ver balanceamento")],
    columns=["checagem", "ok", "detalhe"])
checks.to_csv(f"{OUT_DAT}/validacao.csv", index=False); print(checks.to_string())
print("\nCONCLUSÃO:")
for n_, r in mestre.iterrows():
    print(f"  {n_:3s}: {r.efeito:+.3f} [{r.ic95_inf:+.3f}; {r.ic95_sup:+.3f}] -> {r.veredito}")

                                      checagem     ok            detalhe
0                Shapiro dos resíduos (p>0,05)   True            p=0.807
1                Levene entre células (p>0,05)   True            p=0.119
2           Estabilidade entre seeds (DP<0,01)   True      máx DP=0.0055
3  Concordância IC × ANOVA bruta (sem 'fraco')  False                  B
4            Pre_GPA balanceado (ANOVA p>0,05)   True  ver balanceamento

CONCLUSÃO:
  A  : -0.067 [-0.097; -0.038] -> significativo
  B  : -0.043 [-0.071; -0.013] -> fraco
  C  : +0.195 [+0.167; +0.225] -> significativo
  AB : -0.046 [-0.076; -0.017] -> significativo
  AC : +0.022 [-0.007; +0.052] -> não significativo
  BC : +0.004 [-0.025; +0.034] -> não significativo
  ABC: -0.006 [-0.034; +0.023] -> não significativo
